# v5 Phase 1 Light — Colab 풀버전 실행 노트북

이 노트북은 보험 검색어 이해 엔진을 Colab에서 바로 실행하기 위한 운영용 노트북입니다.

실행 흐름은 다음과 같습니다.

1. 프로젝트 zip 업로드 및 자동 압축 해제
2. 검색어 파일 업로드 또는 샘플 파일 사용
3. `base_rules.xlsx` / `feedback_store.xlsx` 자동 경로 탐지
4. 추론 실행 및 결과 다운로드
5. `review_flag=Y` 우선 검수
6. Colab Quick Fix로 즉시 피드백 반영
7. 재추론 및 수정 반영 확인
8. 업데이트된 결과와 `feedback_store.xlsx` 다운로드

주의: Similarity Matcher는 오탈자/띄어쓰기 보완과 후보 힌트용입니다. 애매한 의미 유사 후보는 `evidence_focus`를 확정하지 않고 `model_hint`와 `review_flag=Y`로 남깁니다.

In [ ]:
# Colab 기본 실행에 필요한 패키지를 설치합니다.
# pandas는 결과 테이블 필터링/display에 사용하고, openpyxl은 xlsx 입출력에 사용합니다.
!pip -q install pandas openpyxl joblib scikit-learn

## 속도 개선 안내

이 노트북은 최신 패치 기준으로 `base_rules.xlsx`를 지정해도 같은 폴더의 `base_rules.csv`를 우선 사용합니다. 또한 `dedupe=True`로 동일 검색어를 한 번만 추론하고, 의미 유사 힌트는 `enable_semantic_hints=False`로 기본 비활성화합니다. 오탈자 보정은 유지됩니다.


In [ ]:
# ============================================================
# 1. 공통 import와 표시 옵션
# ============================================================
import os
import sys
import zipfile
import shutil
from pathlib import Path

import pandas as pd
from google.colab import files
from IPython.display import display

# result가 src.infer.SimpleTable, list[dict], pandas DataFrame 중 무엇으로 오더라도
# 이후 검수/필터링 셀에서 항상 pandas DataFrame으로 다룰 수 있게 변환합니다.
def as_dataframe(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.copy()
    if hasattr(obj, "to_dataframe"):
        return obj.to_dataframe()
    if hasattr(obj, "to_records"):
        return pd.DataFrame(obj.to_records())
    if isinstance(obj, list):
        return pd.DataFrame(obj)
    return pd.DataFrame(obj)

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 80)

WORK_DIR = Path("/content/v5_phase1_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("작업 경로:", WORK_DIR)

## 2. 프로젝트 zip 업로드

아래 셀을 실행한 뒤 `v5_phase1_light_release_*.zip` 파일을 업로드하세요.

이미 `/content/v5_phase1_work/v5_phase1_light` 경로에 프로젝트가 있으면 업로드를 생략해도 됩니다.

In [ ]:
# ============================================================
# 2. 프로젝트 zip 업로드 및 자동 압축 해제
# ============================================================

def upload_project_zip():
    print("프로젝트 zip 파일을 업로드하세요. 예: v5_phase1_light_release_rules_massive_similarity_colab.zip")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("업로드된 파일이 없습니다.")
    zip_names = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
    if not zip_names:
        raise RuntimeError("zip 파일이 업로드되지 않았습니다.")
    zip_path = Path(zip_names[0])
    target_dir = WORK_DIR / "project_unzipped"
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)
    print("압축 해제 완료:", target_dir)
    return target_dir

def find_project_root(base_dir: Path) -> Path:
    candidates = list(base_dir.rglob("src/infer.py"))
    if not candidates:
        raise FileNotFoundError("src/infer.py를 찾지 못했습니다. 프로젝트 zip 구조를 확인하세요.")
    # src/infer.py의 부모의 부모가 프로젝트 루트다.
    return candidates[0].parents[1]

# 업로드를 다시 하지 않고 기존 경로를 쓰고 싶으면 아래 두 줄을 직접 지정하세요.
# project_unzip_dir = Path("/content/v5_phase1_work/project_unzipped")
# PROJECT_ROOT = find_project_root(project_unzip_dir)

project_unzip_dir = upload_project_zip()
PROJECT_ROOT = find_project_root(project_unzip_dir)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("프로젝트 루트:", PROJECT_ROOT)
print("sys.path 등록 완료")

## 3. 검색어 파일 업로드

검색어 파일은 `.xlsx` 또는 `.csv`를 지원합니다.

필수 컬럼은 `query`입니다.  
만약 `query` 컬럼이 없으면 첫 번째 컬럼을 자동으로 `query`로 간주합니다.

In [ ]:
# ============================================================
# 3. 검색어 파일 업로드 또는 샘플 데이터 사용
# ============================================================

def upload_query_file():
    print("검색어 파일을 업로드하세요. xlsx/csv 모두 가능하며 query 컬럼을 권장합니다.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("업로드된 검색어 파일이 없습니다.")
    names = list(uploaded.keys())
    query_files = [n for n in names if n.lower().endswith((".xlsx", ".csv"))]
    if not query_files:
        raise RuntimeError("xlsx 또는 csv 검색어 파일이 없습니다.")
    path = Path(query_files[0])
    print("검색어 파일:", path)
    return path

# 검색어 파일을 직접 업로드하려면 True, 프로젝트 샘플 파일을 쓰려면 False
UPLOAD_QUERY = True

if UPLOAD_QUERY:
    QUERY_PATH = upload_query_file()
else:
    QUERY_PATH = PROJECT_ROOT / "sample_data" / "sample_queries.xlsx"

print("QUERY_PATH =", QUERY_PATH)

## 4. 룰/피드백 경로 자동 탐지

- `base_rules.xlsx`: 기본 룰 사전입니다.
- `feedback_store.xlsx`: Quick Fix 결과가 저장되는 운영 피드백 파일입니다.
- 처음 실행이면 템플릿을 복사해서 새로 만듭니다.
- 기존 피드백을 이어서 쓰려면 업로드 셀에서 기존 `feedback_store.xlsx`를 업로드해 덮어쓰면 됩니다.

In [ ]:
# ============================================================
# 4. base_rules / feedback_store 경로 설정
# ============================================================

BASE_RULES_PATH = PROJECT_ROOT / "configs" / "base_rules.xlsx"
FEEDBACK_TEMPLATE_PATH = PROJECT_ROOT / "configs" / "feedback_store_template.xlsx"
FEEDBACK_STORE_PATH = WORK_DIR / "feedback_store.xlsx"
OUTPUT_DIR = WORK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not BASE_RULES_PATH.exists():
    raise FileNotFoundError(f"base_rules.xlsx를 찾지 못했습니다: {BASE_RULES_PATH}")

# feedback_store가 없으면 템플릿에서 복사합니다.
if not FEEDBACK_STORE_PATH.exists():
    if FEEDBACK_TEMPLATE_PATH.exists():
        shutil.copy(FEEDBACK_TEMPLATE_PATH, FEEDBACK_STORE_PATH)
    else:
        # 템플릿이 없으면 코드가 첫 저장 시 생성할 수 있지만, 운영 편의를 위해 경고만 출력합니다.
        print("feedback_store_template.xlsx가 없습니다. Quick Fix 실행 시 새 파일이 생성됩니다.")

print("BASE_RULES_PATH     =", BASE_RULES_PATH)
print("FEEDBACK_STORE_PATH =", FEEDBACK_STORE_PATH)
print("OUTPUT_DIR          =", OUTPUT_DIR)

## 5. 추론 실행

아래 셀은 전체 검색어를 추론하고 다음 파일을 생성합니다.

- `inference_result.xlsx`
- `review_template.xlsx`

`model_hint`에는 기존 지도학습 모델 값이 아니라, Similarity Matcher가 제안한 오탈자/유사 후보도 함께 기록됩니다.

In [ ]:
# ============================================================
# 5. 추론 실행
# ============================================================

from src.infer import run_inference, export_review_template

RESULT_PATH = OUTPUT_DIR / "inference_result.xlsx"
REVIEW_PATH = OUTPUT_DIR / "review_template.xlsx"

# run_inference는 속도를 위해 내부적으로 가벼운 SimpleTable을 반환할 수 있습니다.
# Colab에서 필터링과 display를 안정적으로 하기 위해 바로 pandas DataFrame으로 변환합니다.
result_table = run_inference(
    query_path=QUERY_PATH,
    base_rules_path=BASE_RULES_PATH,
    feedback_store_path=FEEDBACK_STORE_PATH,
    model_path=None,  # 지도학습 모델은 선택 기능입니다. 기본은 룰 + 유사도 보정입니다.
    output_path=RESULT_PATH,
    enable_semantic_hints=False,  # 유사 의미 힌트는 기본 OFF. 필요할 때만 True로 변경하세요.
    dedupe=True,                 # 동일 query_norm은 한 번만 추론해 속도를 높입니다.
)

review_table = export_review_template(result_table, REVIEW_PATH)

result_df = as_dataframe(result_table)
review_df = as_dataframe(review_table)

print("추론 완료")
print("결과 파일:", RESULT_PATH)
print("리뷰 템플릿:", REVIEW_PATH)
print("결과 타입:", type(result_df), "행 수:", len(result_df))
display(result_df.head(20))

## 6. 검수 대상 우선 확인

`review_flag=Y` 또는 `confidence_flag in ['low', 'needs_review']`인 행을 먼저 봅니다.

오탈자 보정(`match=typo`)은 초기에는 사람이 확인할 수 있도록 `review_flag=Y`로 남깁니다.

`customer_need_detail`은 기존 `customer_need_type` 대분류를 깨지 않고, 질환진단비보장확인/유병력가입가능성확인/청구서류확인처럼 더 세부적인 니즈를 보여주는 보조 컬럼입니다.


In [ ]:
# ============================================================
# 6. 검수 대상 필터링
# ============================================================
# review_flag=Y 또는 confidence_flag가 low/needs_review인 행을 먼저 봅니다.
# 주의: result_df가 SimpleTable/list로 남아 있어도 여기서 다시 DataFrame으로 강제 변환합니다.

result_df = as_dataframe(result_df)

# 일부 예외 상황에서 컬럼이 없으면 빈 컬럼을 만들어 노트북이 멈추지 않게 합니다.
for required_col in ["review_flag", "confidence_flag"]:
    if required_col not in result_df.columns:
        result_df[required_col] = ""

review_targets = result_df[
    (result_df["review_flag"].astype(str).str.upper() == "Y") |
    (result_df["confidence_flag"].astype(str).isin(["low", "needs_review"]))
].copy()

print("전체 행 수:", len(result_df))
print("검수 대상 행 수:", len(review_targets))

cols = [
    "row_id", "query", "query_norm", "gate_type", "insurance_category", "customer_need_type",
    "evidence_focus", "customer_need_detail", "confidence_flag", "review_flag", "model_hint", "evidence_trace"
]
show_cols = [c for c in cols if c in review_targets.columns]
display(review_targets[show_cols].head(50))

## 7. 결과 파일 다운로드

운영자는 우선 결과와 리뷰 템플릿을 다운로드해서 확인할 수 있습니다.

In [ ]:
# ============================================================
# 7. 결과 파일 다운로드
# ============================================================

files.download(str(RESULT_PATH))
files.download(str(REVIEW_PATH))
files.download(str(FEEDBACK_STORE_PATH))

## 8. Colab Quick Fix — 특정 검색어 하나 고정

`quick_fix_exact`는 특정 row_id의 검색어를 사람이 정답으로 고정합니다.

예를 들어 `러이나 치아보험`이 `라이나` 오탈자로 잡혔지만 사람이 확정하고 싶다면 아래처럼 사용합니다.

수정 후에는 `feedback_store.xlsx`의 `exact_overrides`에 저장되고, 다음 재추론부터 LEVEL 0에서 바로 반영됩니다.

In [ ]:
# ============================================================
# 8. quick_fix_exact 예시
# ============================================================
# 아래 값은 예시입니다. 실제로는 위 테이블에서 고칠 row_id를 확인한 뒤 수정하세요.
# 실행하려면 APPLY_EXACT_FIX = True로 바꾸세요.

from src.feedback import quick_fix_exact

APPLY_EXACT_FIX = False

if APPLY_EXACT_FIX:
    quick_fix_exact(
        feedback_store_path=FEEDBACK_STORE_PATH,
        result_df=result_df,
        row_id=0,  # 고칠 row_id
        gate_type="general",
        insurance_category="치아보험",
        customer_need_type="브랜드상품확인",
        evidence_focus="라이나",
        memo="Colab quick_fix_exact 예시"
    )
    print("exact override 반영 완료:", FEEDBACK_STORE_PATH)
else:
    print("예시 셀입니다. 실제 반영하려면 APPLY_EXACT_FIX=True로 변경하세요.")

## 9. Colab Quick Fix — phrase 추가로 일반화

`quick_add_phrase`는 특정 검색어 하나가 아니라 비슷한 검색어까지 잡고 싶을 때 사용합니다.

예: `스케일링`을 `coverage_focus`로 추가하면  
`스케일링 치아보험`, `스케일링 보험`, `치아보험 스케일링`을 다음 실행부터 잡을 수 있습니다.

`use_typo=Y`이면 오탈자 후보 매칭에도 사용됩니다.

In [ ]:
# ============================================================
# 9. quick_add_phrase 예시
# ============================================================

from src.feedback import quick_add_phrase, quick_add_protected_span, quick_add_exclusion

APPLY_PHRASE_FIX = False

if APPLY_PHRASE_FIX:
    quick_add_phrase(
        feedback_store_path=FEEDBACK_STORE_PATH,
        surface="스케일링",
        slot="coverage_focus",
        canonical_value="스케일링보장",
        level=5,
        priority=7500,
        category_hint="치아보험",
        need_hint="보장범위확인",
        gate_hint="detailed",
        is_protected="N",
        match_type="contains",
        memo="스케일링 담보 추가",
        use_typo="Y",
        typo_threshold="0.78",
        use_embedding="Y",
        embedding_text="치아 스케일링 치석 제거 치과 치료 보장",
        embedding_threshold="0.82",
    )
    print("phrase addition 반영 완료:", FEEDBACK_STORE_PATH)
else:
    print("예시 셀입니다. 실제 반영하려면 APPLY_PHRASE_FIX=True로 변경하세요.")

## 10. Colab Quick Fix — protected span / exclusion

- `quick_add_protected_span`: 긴 표현을 먼저 보호해 내부 오탐을 막습니다.
- `quick_add_exclusion`: 특정 surface가 특정 slot으로 해석되지 않게 차단합니다.

예: `국민건강보험`을 protected span으로 추가하면 내부의 `보험`, `환급금`이 민영 보험 문의로 오탐되는 것을 줄일 수 있습니다.

In [ ]:
# ============================================================
# 10. protected span / exclusion 예시
# ============================================================

APPLY_PROTECTION_FIX = False

if APPLY_PROTECTION_FIX:
    quick_add_protected_span(
        feedback_store_path=FEEDBACK_STORE_PATH,
        surface="국민건강보험",
        slot="public_non_private",
        canonical_value="국민건강보험",
        level=2,
        priority=9000,
        need_hint="OOS",
        gate_hint="oos",
        memo="공공보험 protected span"
    )
    quick_add_exclusion(
        feedback_store_path=FEEDBACK_STORE_PATH,
        surface="우리은행",
        block_slot="brand_name",
        reason="은행명이지 보험사 브랜드가 아님"
    )
    print("protected span / exclusion 반영 완료:", FEEDBACK_STORE_PATH)
else:
    print("예시 셀입니다. 실제 반영하려면 APPLY_PROTECTION_FIX=True로 변경하세요.")

## 11. 재추론으로 피드백 반영 확인

Quick Fix를 실행한 뒤 아래 셀을 다시 실행하면 `feedback_store.xlsx`가 반영된 결과를 바로 확인할 수 있습니다.

In [ ]:
# ============================================================
# 11. 피드백 반영 후 재추론
# ============================================================

UPDATED_RESULT_PATH = OUTPUT_DIR / "inference_result_updated.xlsx"
UPDATED_REVIEW_PATH = OUTPUT_DIR / "review_template_updated.xlsx"

updated_df = run_inference(
    query_path=QUERY_PATH,
    base_rules_path=BASE_RULES_PATH,
    feedback_store_path=FEEDBACK_STORE_PATH,
    model_path=None,
    output_path=UPDATED_RESULT_PATH,
)
updated_review_df = export_review_template(updated_df, UPDATED_REVIEW_PATH)

print("재추론 완료")
print("업데이트 결과:", UPDATED_RESULT_PATH)
print("업데이트 리뷰:", UPDATED_REVIEW_PATH)
display(updated_df.head(20))

## 12. 업데이트 파일 다운로드

In [ ]:
# ============================================================
# 12. 업데이트 파일 다운로드
# ============================================================

files.download(str(UPDATED_RESULT_PATH))
files.download(str(UPDATED_REVIEW_PATH))
files.download(str(FEEDBACK_STORE_PATH))

## 13. 대량 리뷰 템플릿 반영

엑셀에서 `review_template.xlsx`를 대량 수정한 경우, 아래 셀로 `feedback_store.xlsx`에 반영할 수 있습니다.

반영 조건:
- `review_status = approved`
- `corrected_*` 컬럼 중 하나 이상 입력

In [ ]:
# ============================================================
# 13. Excel Batch Review 반영
# ============================================================

from src.feedback import compile_review_template

APPLY_BATCH_REVIEW = False

if APPLY_BATCH_REVIEW:
    print("수정된 review_template.xlsx를 업로드하세요.")
    uploaded = files.upload()
    review_files = [name for name in uploaded.keys() if name.lower().endswith(".xlsx")]
    if not review_files:
        raise RuntimeError("수정된 review_template.xlsx 파일이 없습니다.")
    edited_review_path = Path(review_files[0])
    compile_review_template(edited_review_path, FEEDBACK_STORE_PATH)
    print("대량 리뷰 반영 완료:", FEEDBACK_STORE_PATH)
else:
    print("대량 리뷰 반영이 필요하면 APPLY_BATCH_REVIEW=True로 변경하세요.")

## 14. 운영 팁

- `evidence_trace`에 `match=typo`가 보이면 오탈자 보정입니다. 초기에는 검토 후 exact override로 확정하는 것을 권장합니다.
- `model_hint`에 `embedding:...|hint_only=true`가 보이면 의미 유사 후보일 뿐입니다. 자동 확정하지 않습니다.
- `review_flag=Y`가 많은 초기에는 정상입니다. Quick Fix가 쌓일수록 같은 오류는 줄어듭니다.
- 룰을 추가할 때는 단어별 if를 코드에 추가하지 말고 `quick_add_phrase` 또는 `base_rules.xlsx`를 사용하세요.

## 15. 운영용 Quick Fix 일괄 입력 패턴

한 건씩 셀을 복사해 쓰기 번거로우면 아래처럼 리스트로 여러 건을 한 번에 반영할 수 있습니다.
`APPLY_BULK_FIXES=True`로 바꾼 뒤 필요한 행만 남겨 실행하세요.


In [ ]:
# ============================================================
# 15. 운영용 Quick Fix 일괄 입력 패턴
# ============================================================
# 주의:
# - row_id는 result_df 또는 updated_df의 row_id입니다.
# - exact fix는 같은 query_norm에 대해 다음 추론부터 LEVEL 0으로 즉시 적용됩니다.
# - 오탈자 보정 결과를 사람이 확정할 때 사용하면 좋습니다.

APPLY_BULK_FIXES = False

bulk_exact_fixes = [
    # 예시:
    # {
    #     'row_id': 0,
    #     'gate_type': 'general',
    #     'insurance_category': '치아보험',
    #     'customer_need_type': '브랜드상품확인',
    #     'evidence_focus': '라이나',
    #     'memo': '러이나 오탈자 라이나로 확정'
    # },
]

if APPLY_BULK_FIXES:
    for fix in bulk_exact_fixes:
        quick_fix_exact(
            feedback_store_path=FEEDBACK_STORE_PATH,
            result_df=result_df,
            row_id=fix['row_id'],
            gate_type=fix['gate_type'],
            insurance_category=fix['insurance_category'],
            customer_need_type=fix['customer_need_type'],
            evidence_focus=fix.get('evidence_focus', 'null'),
            memo=fix.get('memo', 'bulk quick fix'),
        )
    print('bulk exact fixes 반영 완료:', FEEDBACK_STORE_PATH)
else:
    print('예시 셀입니다. 실행하려면 APPLY_BULK_FIXES=True로 변경하세요.')


## 16. Similarity 결과 해석 가이드

`evidence_trace`와 `model_hint`를 기준으로 자동 확정 가능 여부를 판단합니다.


In [ ]:
# ============================================================
# 16. Similarity 결과 해석 가이드
# ============================================================
# match=typo
#   - 라이나/러이나, 임플란트/임프란트 같은 오탈자 보정입니다.
#   - evidence_focus가 채워질 수 있지만 초기에는 review_flag=Y로 검수합니다.
#
# model_hint에 embedding:...|hint_only=true
#   - 의미 유사 후보입니다.
#   - evidence_focus를 확정하지 않습니다.
#   - 맞다고 판단되면 quick_fix_exact 또는 quick_add_phrase로 승격하세요.

result_df = as_dataframe(result_df)
for required_col in ["evidence_trace", "model_hint"]:
    if required_col not in result_df.columns:
        result_df[required_col] = ""

similarity_cols = ['row_id','query','evidence_focus','evidence_trace','model_hint','confidence_flag','review_flag']
sim_view = result_df[
    result_df['evidence_trace'].astype(str).str.contains('match=typo|compact_contains', regex=True, na=False) |
    result_df['model_hint'].astype(str).str.contains('embedding:', regex=False, na=False)
].copy()
print('Similarity 관련 행 수:', len(sim_view))
display(sim_view[[c for c in similarity_cols if c in sim_view.columns]].head(50))

## 17. 전체 산출물 ZIP 다운로드

모바일 환경이나 여러 파일 다운로드가 불편한 경우, 결과/리뷰/피드백 파일을 하나의 zip으로 묶어 받습니다.


In [ ]:
# ============================================================
# 17. 전체 산출물 ZIP 다운로드
# ============================================================
FINAL_BUNDLE_PATH = WORK_DIR / 'v5_phase1_outputs_bundle.zip'
with zipfile.ZipFile(FINAL_BUNDLE_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.glob('*.xlsx'):
        zf.write(p, arcname=p.name)
    if FEEDBACK_STORE_PATH.exists():
        zf.write(FEEDBACK_STORE_PATH, arcname='feedback_store.xlsx')
print('산출물 번들 생성:', FINAL_BUNDLE_PATH)
files.download(str(FINAL_BUNDLE_PATH))
